# Jour 2 · Mini-projet : analyser un réseau de neurones

> Version corrigée — les activités et leurs explications sont remplies.

## Objectifs

- entraîner un petit CNN avec un protocole reproductible
- interpréter classe, probabilités et confiance
- analyser les erreurs et tester la robustesse à une variation simple


## Mission et fil de l'atelier

Nous reprenons le petit CNN compris précédemment, puis nous examinons ses réponses. La question est : **quand une prédiction est-elle utile, et quand faut-il demander une vérification ?**

Le chemin est **entraîner une référence → lire une sortie → observer les erreurs → essayer un seuil → modifier légèrement les images**. La construction du CNN est fournie pour concentrer cet atelier sur l'analyse. Les activités demandent une modification à la fois et restent exécutables avant d'être complétées.

Avant d'exécuter, expliquez pourquoi il y a dix sorties pour les chiffres. Si vous avez suivi l'extension facultative pair/impair, comparez aussi cette sortie softmax à sa sortie sigmoid ; cette comparaison reste facultative.


## Préparer les outils

Un **import** rend un outil Python disponible. `as np`, `as plt` et `as tf` lui donnent un nom court. NumPy gère les tableaux ; Matplotlib les affiche ; TensorFlow entraîne les réseaux. Les premières lignes configurent les messages et le calcul sur CPU. Elles se recopient telles quelles et ne font pas partie du modèle.

La **graine aléatoire** fixe le point de départ du hasard. `42` est un choix arbitraire : il facilite les comparaisons sans garantir des résultats exactement identiques sur toutes les machines.


In [ ]:
import os  # Réglages de l'environnement avant le chargement de TensorFlow.
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")  # Réduit les messages techniques.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")  # Utilise le CPU pour ce petit cours.

import numpy as np  # Tableaux et calculs numériques.
import matplotlib.pyplot as plt  # Graphiques et images.
import tensorflow as tf  # Réseaux et calcul automatique des gradients.

tf.keras.utils.set_random_seed(42)  # Fixe le hasard pour comparer les expériences.
try:
    # Limite le nombre de threads utilisés par TensorFlow.
    # Un thread par opération suffit ; ce réglage limite la consommation du cours.
    tf.config.threading.set_intra_op_parallelism_threads(1)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except RuntimeError:
    pass  # Ce réglage ne peut plus changer si TensorFlow a déjà commencé à calculer.
print("TensorFlow :", tf.__version__)


## Reprendre les mêmes images, avec trois rôles distincts

`load_digits()` fournit de petites images de chiffres. `astype("float32")` convertit les pixels en nombres décimaux ; `int64` garde les labels sous forme d'entiers. La division par `16` place les pixels entre 0 et 1. Ce maximum vient du jeu de données, pas d'une valeur estimée sur le test.

**Train** : exemples utilisés pour corriger les poids. **Validation** : exemples utilisés pour suivre l'entraînement et choisir les réglages. **Test** : exemples réservés à la mesure finale. `train_test_split` coupe ensemble les images et leurs labels, pour conserver leur association. `stratify=labels` conserve approximativement la proportion de chaque chiffre. `random_state=42` fixe ce découpage.

Dans les noms Python, `X` désigne les entrées et `y` les réponses. Nous séparons d'abord 25 % pour le test, puis 20 % du reste pour la validation. Les proportions finales sont environ 60 % train, 15 % validation et 25 % test.


In [ ]:
from sklearn.datasets import load_digits  # Données incluses localement dans scikit-learn.
from sklearn.model_selection import train_test_split  # Découpage sans mélanger images et labels.

digits = load_digits()
images = digits.images.astype("float32") / 16.0  # Intensités de 0 à 1.
labels = digits.target.astype("int64")  # Chiffre attendu, de 0 à 9.
# Les pixels vont à l'origine de 0 à 16 ; on les ramène entre 0 et 1 pour faciliter l'apprentissage.
# stratify garde à peu près la même proportion de chaque chiffre dans les jeux.
# random_state rend le découpage reproductible.
X_train_val, X_test, y_train_val, y_test = train_test_split(
    images, labels, test_size=0.25, stratify=labels, random_state=42,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.20, stratify=y_train_val, random_state=42,
)
print("Train :", X_train.shape, "Validation :", X_val.shape, "Test :", X_test.shape)
print("Pixels du train :", X_train.min(), "à", X_train.max())


## Ajouter le canal avant d'utiliser Conv2D

Les images sont actuellement rangées en `(nombre, 8, 8)`. `Conv2D` attend `(nombre, hauteur, largeur, canaux)`. Une image en niveaux de gris possède un seul canal : nous ajoutons un axe de taille `1`, sans modifier les pixels.

Dans `X_train[..., np.newaxis]`, `...` conserve tous les axes présents et `np.newaxis` ajoute un nouvel axe à la fin. La forme devient `(nombre, 8, 8, 1)`. Le dernier `1` est le nombre de canaux ; le premier nombre reste le nombre d'images.


In [ ]:
X_train_cnn = X_train[..., np.newaxis]  # Ajoute le canal unique de chaque image.
X_val_cnn = X_val[..., np.newaxis]  # La validation doit avoir exactement le même format.
X_test_cnn = X_test[..., np.newaxis]  # Même transformation pour le test.
print("Avant :", X_train.shape, "Après :", X_train_cnn.shape)
assert X_train_cnn.shape[-1] == 1  # Contrôle : un seul canal de gris. Les pixels sont inchangés.


## Entraîner la même référence simple

Nous reprenons exactement le chemin de base : une convolution, un pooling, Flatten puis dix scores softmax. Nous gardons le même découpage pour pouvoir comparer les observations. Pour reproduire une expérience, on conserve aussi les réglages et les versions des outils.

![Le même CNN simple utilisé comme référence du mini projet](../../ressources/illustrations/jour_02/03_dimensions_simple.svg)


In [ ]:
cnn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(8, 8, 1)),  # Forme d'une image, sans l'axe du batch.
    tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu", name="conv_1"),
    # Les 16 filtres cherchent des motifs locaux ; la hauteur et la largeur restent 8.
    tf.keras.layers.MaxPooling2D(2),  # Maximum de chaque zone 2 × 2 : 8 × 8 devient 4 × 4.
    tf.keras.layers.Flatten(name="features"),  # Réorganise 4 × 4 × 16 en 256 nombres.
    tf.keras.layers.Dense(10, activation="softmax"),  # Un score par chiffre, somme égale à 1.
])
cnn.summary()  # Affiche les formes et le nombre de valeurs apprises par couche.


## Choisir la loss avant de lancer l'entraînement

Le réseau donne dix scores ; la bonne réponse est un entier comme `7`. `sparse_categorical_crossentropy` compare les deux : elle pénalise un score faible pour la bonne classe. **Sparse** signifie ici que le label est un entier, au lieu d'un tableau de dix zéros avec un `1` à la bonne position. **Categorical** indique plusieurs classes exclusives : une image reçoit un seul chiffre.

La **loss** guide l'apprentissage. L'**accuracy** compte seulement la proportion de classes correctement choisies. Une prédiction correcte mais hésitante peut donc avoir la même accuracy qu'une prédiction correcte et nette, tout en ayant une loss différente.

`compile` associe le réseau, la loss et l'optimiseur. `optimizer="adam"` demande à Adam de modifier les poids grâce aux gradients. `metrics=["accuracy"]` ajoute une mesure à afficher ; elle ne remplace pas la loss.


In [ ]:
cnn.compile(
    optimizer="adam",  # Même optimiseur que la démonstration précédente.
    loss="sparse_categorical_crossentropy",  # Labels entiers pour dix classes exclusives.
    metrics=["accuracy"],  # Proportion de chiffres correctement reconnus.
)


## Apprendre, puis suivre la validation

`fit` lance l'entraînement. Une **époque** est un parcours du train ; `batch_size=32` traite 32 images avant une correction des poids. Les images de `validation_data` sont utilisées pour mesurer `val_loss` et `val_accuracy`, jamais pour cette correction.

Un **callback** est une action appelée pendant l'entraînement. `EarlyStopping` observe `val_loss` : après plusieurs époques sans amélioration, il arrête le calcul. `restore_best_weights=True` recharge les meilleurs poids observés sur la validation. `epochs` fixe seulement un maximum ; `verbose=0` cache le journal détaillé.


In [ ]:
arret_anticipe = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True,
)  # Choisit les poids à partir de la validation, pas du test.
history = cnn.fit(
    X_train_cnn, y_train, validation_data=(X_val_cnn, y_val),
    epochs=20, batch_size=32, callbacks=[arret_anticipe], verbose=0,
)
print("Entraînement de la référence terminé.")


## Une sortie probabiliste

Pour chaque image, la couche softmax produit dix valeurs dont la somme vaut environ 1. La classe prédite est celle qui possède la plus grande valeur. Nous appellerons cette valeur maximale **confiance du modèle**.

Cette confiance décrit la préférence interne du réseau entre ses classes. Elle ne constitue pas une garantie : un réseau peut être très confiant et se tromper.


![Deux distributions de confiance sur dix classes](../../ressources/illustrations/jour_02/04_prediction_confiance.png)

*La forme de la distribution aide à distinguer une décision nette d’une décision hésitante.*


## Lire les sorties avant de calculer les classes

`predict` retourne un tableau avec une ligne par image et dix colonnes, une par chiffre. Une **ligne** rassemble les choix pour la même image. `axis=1` demande de travailler sur les colonnes, à l'intérieur de chaque ligne.

`argmax(axis=1)` renvoie la **position** du score le plus élevé : ici, cette position correspond à la classe. `max(axis=1)` renvoie sa **valeur**, appelée confiance du modèle. Exemple : `[0.06, 0.11, 0.70, 0.13]` donne la classe `2` et la confiance `0.70`.

Un score softmax décrit une préférence entre les classes proposées. Il ne garantit pas la vérité de la prédiction, ni le comportement sur une image très différente du train.


![Classe choisie et confiance sont deux lectures différentes du même tableau](../../ressources/illustrations/jour_02/04_argmax_max.svg)


![Confiance Softmax et calibration](../../ressources/illustrations/jour_02/04_confiance_calibration.png)

`probabilities.max(axis=1)` garde le score le plus élevé de chaque ligne ; `argmax(axis=1)` garde l’indice de ce score. Les deux informations répondent à des questions différentes : « quelle classe arrive première ? » et « avec quelle préférence interne ? ».

Un modèle est **bien calibré** si, parmi les prédictions annoncées autour de 80 %, environ 80 % sont effectivement correctes sur un grand nombre de cas comparables. Softmax ne garantit pas automatiquement cette propriété.

Il faut donc éviter de traduire une confiance individuelle de 90 % par « il y a exactement 90 % de chances que ce soit vrai ». Cette valeur reste un signal produit par le modèle et doit être vérifiée sur des données représentatives.


In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix  # Mesures à partir des classes.
import pandas as pd  # Tableau lisible des observations.

probabilities = cnn.predict(X_test_cnn, verbose=0)  # Une ligne de dix scores par image.
predictions = probabilities.argmax(axis=1)  # Classe arrivée première sur chaque ligne.
confidence = probabilities.max(axis=1)  # Valeur du score arrivé premier.
correct = predictions == y_test  # Tableau de True (correct) et False (erreur).
print("Accuracy de référence :", round(accuracy_score(y_test, predictions), 3))
print("Première image : réel", y_test[0], "; prédit", predictions[0], "; confiance", confidence[0])


### Ranger les observations pour les comparer

Un `DataFrame` est un tableau dont les colonnes portent des noms. `results["correct"]` lit sa colonne de réponses correctes. Un **masque booléen** sélectionne les lignes où une condition vaut `True` ; `~` inverse la condition.

`mean()` calcule une moyenne et `head()` montre les premières lignes. Si toutes les prédictions sont correctes, la moyenne sur les erreurs n'est pas définie : on l'indique au lieu d'afficher un nombre trompeur.


In [ ]:
results = pd.DataFrame({
    "réel": y_test, "prédit": predictions,
    "confiance": confidence, "correct": correct,
})  # Une ligne par image, dans le même ordre que X_test.
print("Confiance moyenne si correct :", round(results.loc[correct, "confiance"].mean(), 3))
if (~correct).any():  # .any() demande s'il existe au moins une erreur.
    print("Confiance moyenne si erreur :", round(results.loc[~correct, "confiance"].mean(), 3))
else:
    print("Aucune erreur : pas de moyenne de confiance sur les erreurs.")
display(results.head())  # Les cinq premières lignes, pour relier les colonnes.


## Compter les confusions, puis observer les images

Dans la matrice de confusion, la ligne donne le chiffre réel et la colonne le chiffre prédit. La diagonale compte les réponses correctes. Une case ligne 3, colonne 8 compte les vrais 3 prédits 8. Le tableau numérique est accompagné d'une image colorée : une case plus foncée correspond à un nombre plus élevé.

La matrice répond à « combien ? ». Les images permettent ensuite de demander « qu'est-ce qui a pu tromper le modèle ? ».


In [ ]:
cm = confusion_matrix(y_test, predictions, labels=range(10))  # Réel en lignes, prédit en colonnes.
fig, ax = plt.subplots(figsize=(6, 5))
dessin = ax.imshow(cm, cmap="Blues")  # Une couleur par nombre de cas.
for ligne in range(10):
    for colonne in range(10):
        ax.text(colonne, ligne, str(cm[ligne, colonne]), ha="center", va="center", fontsize=8)
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xlabel("chiffre prédit")
ax.set_ylabel("chiffre réel")
ax.set_title("Les réponses correctes se trouvent sur la diagonale")
fig.colorbar(dessin, ax=ax, label="nombre d'images")
plt.show()


### Donner une catégorie aux erreurs

Une bonne analyse ne se limite pas à afficher des images. On cherche des groupes explicables :

- confusion entre deux classes visuellement proches ;
- image peu lisible ou partiellement effacée ;
- cadrage ou orientation inhabituels ;
- exemple rare dans le train ;
- label potentiellement incorrect.

Cette **taxonomie d’erreurs** transforme une liste de cas ratés en pistes d’action : collecter certains exemples, corriger des labels, modifier la préparation ou revoir la question. Elle évite de changer l’architecture au hasard.


### Choisir des erreurs à afficher

`results.loc[~correct]` sélectionne uniquement les erreurs. `sort_values("confiance", ascending=False)` les trie par confiance décroissante. `head(6)` prend les six premières ; `.index.to_list()` récupère leurs positions dans `X_test`. Ces étapes sont séparées pour voir ce que chacune apporte.

Une erreur très confiante peut indiquer un motif trompeur ou une image atypique. Elle ne prouve pas à elle seule que le label est incorrect.


In [ ]:
erreurs = results.loc[~correct]  # Conserve les lignes où la prédiction est fausse.
erreurs_triees = erreurs.sort_values("confiance", ascending=False)  # Plus confiantes d'abord.
indices_erreurs = erreurs_triees.head(6).index.to_list()  # Positions des images retenues.
if len(indices_erreurs) > 0:
    fig, axes = plt.subplots(1, len(indices_erreurs), figsize=(12, 2.5), squeeze=False)
    for ax, index in zip(axes[0], indices_erreurs):
        ax.imshow(X_test[index], cmap="gray_r")
        ax.set_title(f"réel {y_test[index]} / prédit {predictions[index]} ({confidence[index]:.0%})")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Aucune erreur à afficher pour ce test.")


## Autoriser le modèle à ne pas décider

On peut fixer un seuil minimal de confiance et ne conserver que certaines prédictions. La **couverture** est la proportion d'images pour lesquelles le modèle accepte de répondre. Un seuil élevé réduit généralement la couverture, sans garantir l'absence d'erreurs.


![Couverture, qualité acceptée et test de robustesse](../../ressources/illustrations/jour_02/04_couverture_robustesse.png)

Si le seuil minimal vaut 0,80, une prédiction de confiance 0,72 est refusée. La **couverture** compte la proportion de toutes les images qui restent acceptées.

L’accuracy calculée uniquement sur les prédictions acceptées peut augmenter lorsque le seuil monte, mais ce n’est pas garanti. Il faut toujours présenter ensemble la qualité et la couverture : une accuracy parfaite sur 2 % des cas ne décrit pas un système utile pour les 98 % restants.

Un refus n’est pas nécessairement un échec. Il peut déclencher une vérification humaine ou une autre procédure. Le comportement prévu pour les cas refusés doit être défini avec le besoin métier.


### Du seuil à une proportion

`confidence >= 0.80` produit un masque : `True` pour une réponse acceptée. La **couverture** est la fraction de tous les exemples qui restent acceptés. En calculant la moyenne de ce masque, Python compte `True` comme 1 et `False` comme 0.

Avec dix images dont six acceptées, la couverture vaut 60 %. Si cinq de ces six réponses sont justes, l'accuracy acceptée vaut `5/6`, environ 83 %. Si aucune réponse n'est acceptée, cette accuracy n'est pas définie.

Cet atelier observe les seuils sur le test pour illustrer le compromis. Dans une application, le seuil se choisit sur la validation ; un autre test final resté intact mesure ensuite le système choisi.


## À vous de jouer — essayer un seuil puis lire son effet

Complétez `seuil` avec `0.80`. Lisez ensemble le nombre d'images acceptées, la couverture et l'accuracy acceptée. Pourriez-vous recommander un seuil en ne regardant que l'accuracy acceptée ?


In [ ]:
seuil = 0.80  # Accepte seulement les préférences internes au moins égales à 80 %.
accepte = confidence >= seuil  # Masque des images conservées.
print("Images acceptées :", accepte.sum(), "sur", len(accepte))
print("Couverture :", round(accepte.mean(), 3))
if accepte.any():
    print("Accuracy acceptée :", round(accuracy_score(y_test[accepte], predictions[accepte]), 3))
else:
    print("Aucune image acceptée : accuracy non définie.")


### Lecture du corrigé

Augmenter le seuil conserve moins de réponses. La qualité des réponses restantes peut augmenter, mais il faut toujours annoncer combien sont refusées. Les valeurs exactes dépendent de l'entraînement ; le raisonnement et la manière de calculer les proportions restent les mêmes.


## À vous de jouer — comparer trois seuils sans changer le réseau

Reprenez le calcul précédent pour la liste `[0.60, 0.80, 0.90]`. La boucle est fournie : complétez seulement la liste. Comparez les deux colonnes du résultat avant de proposer un usage métier.


In [ ]:
seuils = [0.60, 0.80, 0.90]  # Trois règles d'acceptation sur les mêmes prédictions.
comparaison = []
for valeur in seuils:
    accepte = confidence >= valeur
    # np.nan signifie « pas de mesure définie » si aucune image n'est acceptée.
    qualite = accuracy_score(y_test[accepte], predictions[accepte]) if accepte.any() else np.nan
    comparaison.append({"seuil": valeur, "couverture": accepte.mean(), "accuracy acceptée": qualite})
display(pd.DataFrame(comparaison))


### Lecture du corrigé

La couverture ne peut qu'être stable ou diminuer quand le seuil monte : la condition devient plus stricte. L'accuracy acceptée, elle, n'est pas forcément monotone. Un seuil n'est utile que si les cas refusés peuvent être traités par une autre procédure, par exemple une vérification humaine.


## Tester une petite variation des données

Un modèle est évalué sur une distribution donnée. Si les images deviennent plus bruitées, plus sombres ou cadrées différemment, sa qualité peut diminuer. Cette expérience ne simule pas tous les changements possibles ; elle rend seulement visible la fragilité potentielle.


## Ajouter du bruit sans changer la réponse attendue

Le **bruit gaussien** ajoute à chaque pixel une petite valeur tirée au hasard, généralement proche de zéro. Son **écart-type** règle l'amplitude habituelle : `0.25` est une perturbation visible pour des pixels normalisés entre 0 et 1. La moyenne zéro signifie que les pixels peuvent augmenter ou diminuer.

`np.random.default_rng(42)` crée un générateur de hasard reproductible. `rng.normal(0, 0.25, taille)` crée un bruit de moyenne 0, d'écart-type 0,25 et de la taille voulue. `np.clip(..., 0, 1)` borne le résultat : `−0.1 → 0`, `0.6 → 0.6`, `1.2 → 1`.

Le label reste le même, mais une perturbation trop forte peut rendre le chiffre illisible. Nous afficherons d'abord une image avant/après, puis mesurerons le modèle. Ce test décrit une seule variation contrôlée, pas toutes les situations réelles.


## À vous de jouer — observer puis mesurer une variation

Complétez uniquement `amplitude_bruit` avec `0.25`. Avant de lire les scores, regardez les deux images. Le modèle n'est pas réentraîné : nous comparons ses réponses aux images originales et bruitées.


In [ ]:
amplitude_bruit = 0.25  # Perturbation visible sur l'échelle de pixels [0, 1].
rng = np.random.default_rng(42)
bruit = rng.normal(0, amplitude_bruit, X_test_cnn.shape)  # Même forme que les images.
X_noisy = np.clip(X_test_cnn + bruit, 0, 1).astype("float32")  # Garde les valeurs valides.


### Lecture du corrigé

L'ajout de bruit conserve la forme `(nombre, 8, 8, 1)`, mais change les pixels. La baisse d'accuracy éventuelle montre une fragilité à cette variation précise. Une confiance moyenne élevée ne compense pas des réponses fausses.


In [ ]:
if X_noisy is not None:
    fig, axes = plt.subplots(1, 2, figsize=(5, 2.5))
    axes[0].imshow(X_test[0], cmap="gray_r", vmin=0, vmax=1)  # Même échelle de gris.
    axes[1].imshow(X_noisy[0, :, :, 0], cmap="gray_r", vmin=0, vmax=1)
    axes[0].set_title("originale")
    axes[1].set_title("bruitée")
    for ax in axes:
        ax.axis("off")
    plt.show()


In [ ]:
if X_noisy is not None:
    noisy_probabilities = cnn.predict(X_noisy, verbose=0)  # Même CNN, nouveaux pixels.
    noisy_predictions = noisy_probabilities.argmax(axis=1)  # Classe pour chaque image bruitée.
    print("Accuracy originale :", round(accuracy_score(y_test, predictions), 3))
    print("Accuracy bruitée :", round(accuracy_score(y_test, noisy_predictions), 3))
    print("Confiance originale :", round(float(confidence.mean()), 3))
    print("Confiance bruitée :", round(float(noisy_probabilities.max(axis=1).mean()), 3))


## Donner une conclusion proportionnée aux observations

Complétez oralement ou par écrit : « sur ces images de test, le modèle obtient… ; les erreurs visibles concernent… ; avec le seuil…, il répond sur… des images ; avec le bruit testé, la qualité… ».

Une amélioration se choisit selon la cause observée : données plus représentatives, meilleure qualité d'image, augmentation adaptée ou procédure de refus. Changez une chose à la fois et comparez la validation avant de refaire une évaluation finale.

Si les activités ne sont pas encore remplies, le résultat disponible concerne seulement la référence et ses erreurs. Les effets des seuils et du bruit deviennent des observations quand vous exécutez ces activités.
